# Mistral v0.3

This notebook use Mistral v0.3 for infering if texts are written by LLM. There is no finetuning performed. The model will be tested in 0-shot and 5-shot prompting.

# Load model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from transformers import logging
logging.set_verbosity_warning()
logging.set_verbosity_error()

import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import tqdm

device = "cuda" # the device to load the model onto

token = ''
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3", token=token, torch_dtype=torch.float16)
model.to(device)

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3", token=token, torch_dtype=torch.float16)

# Load data

In [ ]:
dataset = 'SQUAD_GPT4'

import pandas as pd
import numpy as np

data = pd.read_csv(f'Data/{dataset}.csv', encoding = "ISO-8859-1")
data.head()

# Zero-shot Prompting

In [ ]:
f1s = []

for seed in [1111, 2222,3333,4444,5555,6666,7777,8888,9999,1010]:
    np.random.seed(seed)
    print(seed)
    sampling_indices = np.random.uniform(0,1,data.shape[0])
    train_data = data.loc[sampling_indices < 0.9]
    test_data = data.loc[sampling_indices >= 0.9]
    train_data = pd.DataFrame({'text':pd.concat([train_data['answer'], train_data['gpt1'], train_data['gpt2']]),
                               'label':[0]*train_data.shape[0] + [1]*train_data.shape[0]*2})
    test_data = pd.DataFrame({'text':pd.concat([test_data['answer'], test_data['gpt2']]),
                               'label':[0]*test_data.shape[0]+[1]*test_data.shape[0]})

    outs = []
    for text in tqdm.tqdm(test_data['text']):
        first_mes = 'act as an AI content detector, determine whether the text wrapped in triple quotes """<text>""" was written by human or not. answer "yes" or "no" only.'
        messages = [
            {'role': 'user', 'content': f'{first_mes} """{text}"""'}
        ]

        encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

        model_inputs = encodeds.to(device)

        generated_ids = model.generate(model_inputs, max_new_tokens=2, do_sample=True)
        decoded = tokenizer.batch_decode(generated_ids)
        outs.append(decoded[0].split('[/INST]')[-1].replace('</s>',''))

    outs_bin = [0 if 'yes' in s.lower() else 1 for s in outs]
    f1s.append(f1_score(test_data['label'], outs_bin))
    
np.mean(f1s)

# Five-shot Prompting

In [ ]:
f1s = []
for seed in [1111, 2222,3333,4444,5555,6666,7777,8888,9999,1010]:
    np.random.seed(seed)
    print(seed)
    #split
    sampling_indices = np.random.uniform(0,1,data.shape[0])
    train_data = data.loc[sampling_indices < 0.9]
    test_data = data.loc[sampling_indices >= 0.9]
    train_data = pd.DataFrame({'text':pd.concat([train_data['answer'], train_data['gpt1'], train_data['gpt2']]),
                               'label':[0]*train_data.shape[0] + [1]*train_data.shape[0]*2})
    test_data = pd.DataFrame({'text':pd.concat([test_data['answer'], test_data['gpt2']]),
                               'label':[0]*test_data.shape[0]+[1]*test_data.shape[0]})

    #run model
    outs = []
    for text in tqdm.tqdm(test_data['text']):
        #get examples
        r = train_data.sample()
        first_mes = 'act as an AI content detector, determine whether the text wrapped in triple quotes """<text>""" was written by human or not. answer "yes" or "no" only.'
        messages = [
            {'role': 'user', 'content': f'{first_mes} """{r["text"].values[0]}"""'},
            {'role': 'assistant', 'content' : f'{"yes" if r["label"].values[0]==0 else "no"}'}
        ]
        for r in train_data.sample(5).iterrows():
            messages.append({'role': 'user', 'content': f'"""{r[1]["text"]}"""'})
            messages.append({'role': 'assistant', 'content' : f'{"yes" if r[1]["label"]==0 else "no"}'})
        #add input text
        messages.append({'role': 'user', 'content': f'"""{text}"""'})

        encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")

        model_inputs = encodeds.to(device)

        generated_ids = model.generate(model_inputs, max_new_tokens=2, do_sample=True)
        decoded = tokenizer.batch_decode(generated_ids)
        outs.append(decoded[0].split('[/INST]')[-1].replace('</s>',''))

    outs_bin = [0 if 'yes' in s.lower() else 1 for s in outs]
    f1s.append(f1_score(test_data['label'], outs_bin))

np.mean(f1s)